<a href="https://colab.research.google.com/github/LuizWalker/Codigos_TCC/blob/main/Algoritmo_Janelamento_e_Salvamento_Imagens_Fraturadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Acesso ao drive

In [ ]:
from google.colab import drive #Para importar do drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Instalação e Import das Bibliotecas

In [ ]:
#Pacotes C++ necessárias pra o funcionamento das bibliotecas pra leitura das imagens que precisavam ser montadas no formato JPEG.
!apt-get install -y libgdcm-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgdcm-dev is already the newest version (3.0.10-1build2).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
#Instalação das bibliotecas necessárias

!pip install --upgrade pydicom
!pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg gdcm

try:
    import pydicom
    import gdcm
    import pylibjpeg
    print(f"pydicom version: {pydicom.__version__}")
    print(f"gdcm version: {gdcm.Version.GetVersion()}")
    print(f"pylibjpeg version: {pylibjpeg.__version__}")
    print("Sucesso: Todas as bibliotecas foram importadas corretamente.")
except Exception as e:
    print(f"Erro na verificação das bibliotecas: {e}")

Erro na verificação das bibliotecas: No module named '_gdcmswig'


In [ ]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
import os

#Função do Janelamento

In [ ]:
def janelamento(imagem, distanciamento_janela, centro_janela):

  imagem_float = imagem.astype(float) #conveter para float evitando problemas de arredondamento

  valor_minimo = centro_janela - (distanciamento_janela/2) #Limite inferior da janela
  valor_maximo = centro_janela + (distanciamento_janela/2) #Limite superior da janela

  imagem_float[imagem_float < valor_minimo] = valor_minimo #Valores abaixo do limite inferior da janela são setados pro valor mínimo
  imagem_float[imagem_float > valor_maximo] = valor_maximo #Valores acima do limite superior da janela são setados pro valor mínimo

  imagem_float = ((imagem_float - valor_minimo) / (valor_maximo - valor_minimo)) * 255 #Mapeia os valores para a faixa de 0 a 255

  return imagem_float.astype(np.uint8)

##Valores para o janelamento

In [ ]:
# Valores empíricos para realçar os ossos

distanciamento_janela_ossos = 2000  #Valores citados nas referências bibliográficas
centro_janela_ossos = 500

#distanciamento_janela_ossos = 500
#centro_janela_ossos = 1100

#Algoritmo para realizar o janelamento de todas as subpastas

In [ ]:
root_folder = "/content/drive/MyDrive/Dataset_Fraturas/Train_images_fraturados_slices_separados" #Lembrar de modificar

output_folder = "/content/drive/MyDrive/Imagens_Processadas_TCC/Pacientes_Fraturados_PNG" #Lembrar de modificar


# 1. Cria a pasta de saída principal se ela não existir
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Pasta de destino '{output_folder}' criada.")

print("Iniciando janelamento e salvamento...")

# 2. Itera sobre as pastas de pacientes dentro da pasta raiz
for patient_folder in os.listdir(root_folder):
    patient_folder_path = os.path.join(root_folder, patient_folder)

    # Ignora arquivos que não são pastas (ex: o próprio DICOM)
    if not os.path.isdir(patient_folder_path):
        continue

    # 3. Cria uma pasta de saída para este paciente
    output_patient_folder = os.path.join(output_folder, patient_folder)
    if not os.path.exists(output_patient_folder):
        os.makedirs(output_patient_folder)

    print(f"Processando paciente: {patient_folder}")

    # 4. Itera sobre os arquivos DICOM dentro da pasta do paciente
    for filename in os.listdir(patient_folder_path):
        if filename.endswith('.dcm'):
            file_path = os.path.join(patient_folder_path, filename)

            try:
                ds = pydicom.dcmread(file_path)
                pixel_data = ds.pixel_array

                janelamento_ossos = janelamento(pixel_data, distanciamento_janela_ossos, centro_janela_ossos)

                # 5. Salva a imagem janelada na pasta do paciente
                output_filename = filename.replace('.dcm', '.png')
                output_path = os.path.join(output_patient_folder, output_filename)

                plt.imsave(output_path, janelamento_ossos, cmap='gray')
                print(f"  - Imagem '{output_filename}' salva com sucesso.")

            except Exception as e:
                print(f"  - Erro ao processar o arquivo '{filename}': {e}")

print("Processamento concluído.")

A saída de streaming foi truncada nas últimas 5000 linhas.
  - Imagem '98.png' salva com sucesso.
  - Imagem '269.png' salva com sucesso.
  - Imagem '263.png' salva com sucesso.
  - Imagem '271.png' salva com sucesso.
  - Imagem '262.png' salva com sucesso.
  - Imagem '267.png' salva com sucesso.
Processando paciente: 1.2.826.0.1.3680043.11644
  - Imagem '197.png' salva com sucesso.
  - Imagem '195.png' salva com sucesso.
  - Imagem '200.png' salva com sucesso.
  - Imagem '196.png' salva com sucesso.
  - Imagem '202.png' salva com sucesso.
  - Imagem '201.png' salva com sucesso.
  - Imagem '199.png' salva com sucesso.
  - Imagem '198.png' salva com sucesso.
  - Imagem '212.png' salva com sucesso.
  - Imagem '204.png' salva com sucesso.
  - Imagem '206.png' salva com sucesso.
  - Imagem '207.png' salva com sucesso.
  - Imagem '203.png' salva com sucesso.
  - Imagem '205.png' salva com sucesso.
  - Imagem '210.png' salva com sucesso.
  - Imagem '209.png' salva com sucesso.
  - Imagem '21